In [ ]:
import pandas as pd
import numpy as np
import pickle
import os


from google.colab import drive
pd.set_option('display.max_columns', None)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:

# 2. Define base path correctly
base_path = '/content/drive/MyDrive/UFRGS/ESWA'

# drive.mount('/content/drive')
df = pd.read_excel(base_path+'/gpt4_cot_labes_revisada_final_fp.xlsx',sheet_name='NOVO')

In [ ]:
df

,trecho,prompt,score,topic_num,Name,ref,tipo,tipo2
0,vai usar lily roupa da planet tommy joias loui...,0,0,0,0_de_moda_marcas_consumismo,1,manual,manual
1,"Mais se os manos são do bom, bota o puma disc ...",0,0,0,0_de_moda_marcas_consumismo,1,manual,manual
2,"Na garagem um Camaro, uma Hornet\nCordão de ou...",0,0,0,0_de_moda_marcas_consumismo,1,manual,manual
3,Tá de lancha e as gata no iate\nPaco Rabanne e...,0,0,0,0_de_moda_marcas_consumismo,1,manual,manual
4,"O que eles têm, nós têm em dobro\nNós têm tant...",0,0,0,0_de_moda_marcas_consumismo,1,manual,manual
...,...,...,...,...,...,...,...,...
486,Vim de uma favela onde nunca foi fácil\nTodos ...,0,0,11,0,1,revisao_fp_at_least_2,manual
487,Você me conquistou apenas com um sorriso\nVocê...,0,0,3,0,1,revisao_fp_at_least_2,manual
488,Você me conquistou\nMe fez louco de amor\nVocê...,0,0,2,0,1,revisao_fp_at_least_2,manual
489,"Vodka ou água de coco, pra mim tanto faz Eu go...",0,0,3,0,1,revisao_fp_at_least_2,manual


In [ ]:
df['trecho'].nunique()

218

{'0': {'descricao': 'Consumo e ostentação de produtos de luxo, marcas famosas; estilo de vida luxoso',
  'positivos': ['atencao olha o ritmo quer cordao de ouro importado e um carrao ela da pra nois que nois e patrao ela da pra nois que nois e patrao ela da pra nois que nois e patrao',
   'abre espaco pros cara do momento que mete joga dentro e faz voce se apaixonar de abercrombie de christian ou ed hardy de anel uma aeropostale os cara vao pirar babar',
   'É os mente blindada da quebrada que age sem emoção\nAnda de Audi, Hornet, de Juliet\nE as novinha morre do coração\nTem que ser zika da balada\nOuro e prata destaca o aparelho\nAndar de Lacoste, de Brooksfield\nE o gelo acompanha dama de vermelho'],
  'negativos': ['Atitude, disposição só tem mesmo quem é\nNa selva urbana sobrevive só com muita fé\nÉ o som do morro que hoje invade o asfalto\nO som dos carros é o tamborzão que toca alto\nTem favelado passeando de carro importado\nDe Meriva ou de Corolla ou Honda envenenado\nCria do 

In [ ]:

dic = {  0: 'Consumo e ostentação de produto de luxos, marcas famosas; estil de vida luxoso',
  1: 'relacionamentos amorosos, intimidade, amor, paixão',
  2: 'superação de obstáculos na vida; reflexão sobre a vida e suas dificuldades; desafios e sucessos',
  3: 'desejo sexual; sedução; sexualidade; atração física; expressões vulgares',
  4: 'festa; ambientes noturnos; jovens em festas',
  5: 'consumo de álcool e drogas; consumo de substâncias',
  6: 'violência; linguagem violento; linguagem vulgar e violento',
  7: 'relações familiares; relação mãe e filho; maternidade',
  9: 'mulheres; estereótipos e aparência das mulheres',
  11: 'Vida em favela; bairros e favelas; cultura e favelas brasileiras',
  12: 'arrependimento e pedidio de perdão; arrependimento religioso; arrpendimento e pedido de perdão aos pais'
}


In [ ]:
pip install -q transformers accelerate bitsandbytes sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.1 MB/s eta 0:00:00


In [ ]:
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_id = "meta-llama/Meta-Llama-3-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)


config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/51.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

In [ ]:
def get_response(messages):


  input_ids = tokenizer.apply_chat_template(
      messages,
      add_generation_prompt=True,
      return_tensors="pt"
  ).to(model.device)

  terminators = [
      tokenizer.eos_token_id,
      tokenizer.convert_tokens_to_ids("<|eot_id|>")
  ]

  outputs = model.generate(
      input_ids,
      max_new_tokens=10,
      eos_token_id=terminators,
      do_sample=True,
      temperature=0.2,
      top_p=0.9,
  )
  response = outputs[0][input_ids.shape[-1]:]
  return tokenizer.decode(response, skip_special_tokens=True)

In [ ]:
few_shot_prompt = {
  "0": {
    "descricao": "Consumo e ostentação de produtos de luxo, marcas famosas; estilo de vida luxoso",
    "positivos": [
      "atencao olha o ritmo quer cordao de ouro importado e um carrao ela da pra nois que nois e patrao ela da pra nois que nois e patrao ela da pra nois que nois e patrao",
      "abre espaco pros cara do momento que mete joga dentro e faz voce se apaixonar de abercrombie de christian ou ed hardy de anel uma aeropostale os cara vao pirar babar",
      "É os mente blindada da quebrada que age sem emoção\nAnda de Audi, Hornet, de Juliet\nE as novinha morre do coração\nTem que ser zika da balada\nOuro e prata destaca o aparelho\nAndar de Lacoste, de Brooksfield\nE o gelo acompanha dama de vermelho"
    ],
    "negativos": [
 "Eu vou lutar até o fim\nVou trazer você pra mim\nE eu te chamo bem assim",
  "Amigo, eu moro na favela, sim, senhor\nNão tenho vergonha de lá viver\nNós somos pobres, mas também temos direito\nDe ser um povo satisfeito e sem sofrer",
  "mundao girou mundao girou e as contas de casa nos quitou deus me abencoou deus abencoou eu pedi perdao pra ela e ela me perdoou"    ]
  },
  "1": {
    "descricao": "relacionamentos amorosos, intimidade, amor, paixão",
    "positivos": [
      "Eu vou lutar até o fim\nVou trazer você pra mim\nE eu te chamo bem assim",
      "Eu gosto de você\nEu gosto de te amar\nMas se vai ficar bebendo\nNosso amor vai acabar",
      "Avassalador\nUm cara interessante\nEsculacho o teu amante\nAté o teu ficante"
    ],
    "negativos": [
    "Amigo, eu moro na favela, sim, senhor\nNão tenho vergonha de lá viver",
    "Open droga na favela\nPras que senta com o popô\nEssa bandida tá de brincadeira\nDois quilo de erva o PH sorteou",
    "É os mente blindada da quebrada que age sem emoção\nAnda de Audi, Hornet, de Juliet"
    ]
  },
  "2": {
    "descricao": "superação de obstáculos na vida; reflexão sobre a vida e suas dificuldades; desafios e sucessos",
    "positivos": [
      "mc neguinho do kaxeta e que faz um tempo que eu nao veja a minha coroa prometi pra ela que ia fazer o jogo virar trampo atras de trampo eles acham que eu to a toa ainda vao falar que e sorte quando os meus trampo vingar e que faz um tempo que eu nao veja a minha coroa prometi pra ela que ia fazer o jogo virar trampo atras de trampo eles acham que eu to a toa ainda vao falar que e sorte quando os meus trampo vingar",
      "Moro num lugar maravilhoso\nOnde todos tem coragem de lutar para vencer\nPara vencer\nMoro na favela agitada\nOnde a rapaziada unida pra valer\nVem ver para crer\nVem vem",
      "O ano todo, a vida aqui muda demais\nCom a tristeza querendo atormentar\nÉ criancinha precisando de atenção\nA mãe sofrendo faz encrenca com o patrão\nAqui a gente sempre luta por melhor\nMas a sociedade leva a gente pra pior\nDesempregado se acabando embriagado\nAlguns se matam vendo a vida piorar"
    ],
    "negativos": [
        "Eu puxo o seu cabelo\nFaço o que você gosta\nDou tapa na bundinha vou de frente, vou de costas.",
    "Eu e minha namorada a gente tava agarradinho\nAté que o meu amor bebeu um copo de vinho",
    "atencao olha o ritmo quer cordao de ouro importado e um carrao"]
  },
  "3": {
    "descricao": "desejo sexual; sedução; sexualidade; atração física; expressões vulgares",
    "positivos": [
      "Mina cheia de marra de bam\nSe eu te pego eu te escangalho\nNa hora do prazer sou eu que faço o trabalho\nFica cheia de caozada\nFalando que eu sou de bobeira\nMas lá no cativeiro\nSou eu que pego à noite inteira",
      "Eu puxo o seu cabelo\nFaço o que você gosta\nDou tapa na bundinha vou de frente, vou de costas.",
      "Piercing na barriguinha,\ncintura marcadinha,\nolha pra essa mina,\nmano mais que tentação,\nquando ela passa,\ngeral fica bolado,\naté o dj tá danadão."
    ],
    "negativos": [
    "Eu vou lutar até o fim\nVou trazer você pra mim\nE eu te chamo bem assim",
    "Humildemente a gente vai manda assim\nEu sou MC Naldinho e eu sou Renato MC\nEntre morros e favelas vou falar",
    "mundao girou mundao girou e as contas de casa nos quitou deus me abencoou"   ]
  },
  "4": {
    "descricao": "festa; ambientes noturnos; jovens em festas",
    "positivos": [
      "Mina cheia de marra de bam\nSe eu te pego eu te escangalho\nNa hora do prazer sou eu que faço o trabalho\nFica cheia de caozada\nFalando que eu sou de bobeira\nMas lá no cativeiro\nSou eu que pego à noite inteira",
      "Eu e minha namorada a gente tava agarradinho\nAté que o meu amor bebeu um copo de vinho\nEla perde o controle\nE fica logo animadinha\nTá todo mundo olhando pra minha patricinha\nTá chapada, tá doidona\nTá descendo até o chão\nJá tô pagando mico, olha que situação",
      "E se hoje tem baile funk DJ Colombia no sampa e representa\nMC B.O na ideologia na putaria da cena arrebenta\nOlha o bonde ta passando e a galera grita ei\nKit da Onbongo, da Quilksilver, Wilk, Lacoste e também Polo Play\nOakley da cabeça aos pés Puma Disk é passado\nDalbuseiri ouro, dalva besouro"
    ],
    "negativos": [
      "Sua mulher segura o filho em sua casa\nFazendo lista pra fazer seu funeral\nO filho cresce analfabeto e sem infância\nAi, minha gente, quem trabalha e a criança\nAlguns aceitam, outros ficam revoltados\nE na cabeça o mal começa a rondar\nAlguns amigos vendo ele se envolver\nChamam no conselho e eles aceitam com prazer",
      "Eu gosto de você\nEu gosto de te amar\nMas se vai ficar bebendo\nNosso amor vai acabar",
       "Moro num lugar maravilhoso\nOnde todos tem coragem de lutar para vencer"
    ]
  },
  "5": {
    "descricao": "consumo de álcool e drogas; consumo de substâncias",
    "positivos": [
      "O ano todo, a vida aqui muda demais\nCom a tristeza querendo atormentar\nÉ criancinha precisando de atenção\nA mãe sofrendo faz encrenca com o patrão\nAqui a gente sempre luta por melhor\nMas a sociedade leva a gente pra pior\nDesempregado se acabando embriagado\nAlguns se matam vendo a vida piorar",
      "Eu e minha namorada a gente tava agarradinho\nAté que o meu amor bebeu um copo de vinho\nEla perde o controle\nE fica logo animadinha\nTá todo mundo olhando pra minha patricinha\nTá chapada, tá doidona\nTá descendo até o chão\nJá tô pagando mico, olha que situação",
      "Open droga na favela\nPras que senta com o popô\nEssa bandida tá de brincadeira\nDois quilo de erva o PH sorteou"
    ],
    "negativos": [
      "Sua mulher segura o filho em sua casa\nFazendo lista pra fazer seu funeral\nO filho cresce analfabeto e sem infância\nAi, minha gente, quem trabalha e a criança\nAlguns aceitam, outros ficam revoltados\nE na cabeça o mal começa a rondar\nAlguns amigos vendo ele se envolver\nChamam no conselho e eles aceitam com prazer",
      "Eu gosto de você\nEu gosto de te amar\nMas se vai ficar bebendo\nNosso amor vai acabar",
      "Comentou do black lança exibido do pai\nAi ai ai ai\nAi ai ai ai\nNo coldre a glock 40 deixa os menor sagaz"
    ]
  },
  "6": {
    "descricao": "violência; linguagem violenta; linguagem vulgar e violenta",
    "positivos": [
      "Mina cheia de marra de bam\nSe eu te pego eu te escangalho\nNa hora do prazer sou eu que faço o trabalho\nFica cheia de caozada\nFalando que eu sou de bobeira\nMas lá no cativeiro\nSou eu que pego à noite inteira",
      "Eu puxo o seu cabelo\nFaço o que você gosta\nDou tapa na bundinha vou de frente, vou de costas.",
      "Sua mulher segura o filho em sua casa\nFazendo lista pra fazer seu funeral\nO filho cresce analfabeto e sem infância\nAi, minha gente, quem trabalha e a criança\nAlguns aceitam, outros ficam revoltados\nE na cabeça o mal começa a rondar\nAlguns amigos vendo ele se envolver\nChamam no conselho e eles aceitam com prazer"
    ],
    "negativos": [
      "Eu gosto de você\nEu gosto de te amar\nMas se vai ficar bebendo\nNosso amor vai acabar",
      "Eu vou lutar até o fim\nVou trazer você pra mim\nE eu te chamo bem assim",
      "O ano todo, a vida aqui muda demais\nCom a tristeza querendo atormentar\nÉ criancinha precisando de atenção\nA mãe sofrendo faz encrenca com o patrão\nAqui a gente sempre luta por melhor\nMas a sociedade leva a gente pra pior\nDesempregado se acabando embriagado\nAlguns se matam vendo a vida piorar"
    ]
  },
  "7": {
    "descricao": "relações familiares; relação mãe e filho; maternidade",
    "positivos": [
      "o mae olha que fita eu to num som com o hariel lembra que eu te falei que isso aqui ia virar claro que te entendia eu sim te buscaria independente da familia sim ou nao acreditar mas fala mais um pouco que eu amo sua historia na real me pergunto como aguentou isso tudo felipe ce nunca teve que entender uma mulher so saiba que sem ela ce nao vai durar muito no mundo",
      "O ano todo, a vida aqui muda demais\nCom a tristeza querendo atormentar\nÉ criancinha precisando de atenção\nA mãe sofrendo faz encrenca com o patrão\nAqui a gente sempre luta por melhor\nMas a sociedade leva a gente pra pior\nDesempregado se acabando embriagado\nAlguns se matam vendo a vida piorar",
      "Mais é por isso que eu te falo,\nMinha família acordou,\nEu quando levantei da cama logo a minha filha chorou,\nEla falou assim papai,\nEu falei filha não fica com medo,\nMais ela me perguntou."
    ],
    "negativos": [
      "ai hari nao esquece de mandar um beijo pra sua mae nao hein mano mas ai tia karen se ele esquecer eu ja vou mandar aqui ta um beijo meu amor fica com deus ta que ogum proteja seu filho sua familia sua casa certo que o papai oxossi tambem proteja sua filha que nos e o memo santo ne ta ligado",
      "Eu vou lutar até o fim\nVou trazer você pra mim\nE eu te chamo bem assim",
      "Eu gosto de você\nEu gosto de te amar\nMas se vai ficar bebendo\nNosso amor vai acabar"
    ]
  },
  "9": {
    "descricao": "mulheres; estereótipos e aparência das mulheres",
    "positivos": [
      "Agora é ruim de tu fugir\nQue o tigrão vai te engolir\nSe tu corre por aqui\nEu te pego logo ali",
      "Eu puxo o seu cabelo\nFaço o que você gosta\nDou tapa na bundinha vou de frente, vou de costas.",
      "Mina cheia de marra de bam\nSe eu te pego eu te escangalho\nNa hora do prazer sou eu que faço o trabalho\nFica cheia de caozada\nFalando que eu sou de bobeira\nMas lá no cativeiro\nSou eu que pego à noite inteira"
    ],
    "negativos": [
      "Eu vou lutar até o fim\nVou trazer você pra mim\nE eu te chamo bem assim",
      "Ih dasqui dasqui dasqui ih\nEu sou a dona gigi\nIh dasqui dasqui dasqui ih\nEsse aqui é meu esposo\nIh dasqui dasqui dasqui ih\nEsse aí é seu esposo?!?\nIh dasqui dasqui dasqui ih\nÉ sim",
      "Avassalador\nUm cara interessante\nEsculacho o teu amante\nAté o teu ficante"
    ]
  },
  "11": {
    "descricao": "Vida em favela; bairros e favelas; cultura e favelas brasileiras",
    "positivos": [
      "Humildemente a gente vai manda assim\nEu sou MC Naldinho e eu sou Renato MC\nEntre morros e favelas vou falar\nA realidade pois agora vai escutar",
      "Amigo, eu moro na favela, sim, senhor\nNão tenho vergonha de lá viver\nNós somos pobres, mas também temos direito\nDe ser um povo satisfeito e sem sofrer",
      "Amigo, eu moro moro moro\nNão tenho vergonha de lá viver\nNós somos pobres, mas também temos direito\nDe ser um povo satisfeito e sem sofrer"
    ],
    "negativos": [
      "Avassalador\nUm cara interessante\nEsculacho o teu amante\nAté o teu ficante",
      "Eu vou lutar até o fim\nVou trazer você pra mim\nE eu te chamo bem assim",
      "A minha familia ta chorando agora ta passando mal,\nSerá que vamos curtir o ano novo,\nE curtir nosso natal."
    ]
  },
  "12": {
    "descricao": "arrependimento e pedido de perdão; arrependimento religioso; arrependimento e pedido de perdão aos pais",
    "positivos": [
      "mundao girou mundao girou e as contas de casa nos quitou deus me abencoou deus abencoou deus abencoou eu pedi perdao pra ela e ela me perdoou eu pedi perdao pra ela e ela me perdoou deus abencoou deus abencoou deus abencoou",
      "A minha familia ta chorando agora ta passando mal,\nSerá que vamos curtir o ano novo,\nE curtir nosso natal.",
      "O mundo deu voltas e as despesas domésticas foram pagas, Deus me abençoou. Pedi perdão a ela e ela me perdoou, Deus abençoou."
    ],
    "negativos": [
      "Eu vou lutar até o fim\nVou trazer você pra mim\nE eu te chamo bem assim",
      "Eu gosto de você\nEu gosto de te amar\nMas se vai ficar bebendo\nNosso amor vai acabar",
      "Avassalador\nUm cara interessante\nEsculacho o teu amante\nAté o teu ficante"
    ]
  }
}

import torch

def get_response(messages):
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt"
    )
    input_ids = inputs["input_ids"].to(model.device)
    attention_mask = inputs.get("attention_mask")
    if attention_mask is not None:
        attention_mask = attention_mask.to(model.device)

    terminators = [tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids("<|eot_id|>")]

    outputs = model.generate(
        input_ids,
        attention_mask=attention_mask,
        max_new_tokens=10,
        eos_token_id=terminators,
        do_sample=True,
        temperature=0.2,
        top_p=0.9,
    )
    return tokenizer.decode(outputs[0][input_ids.shape[-1]:], skip_special_tokens=True)

descricao_topicos = {
    0:  "Consumo e ostentação de produto de luxos, marcas famosas; estil de vida luxoso",
    1:  "relacionamentos amorosos, intimidade, amor, paixão",
    2:  "superação de obstáculos na vida; reflexão sobre a vida e suas dificuldades; desafios e sucessos",
    3:  "desejo sexual; sedução; sexualidade; atração física; expressões vulgares",
    4:  "festa; ambientes noturnos; jovens em festas",
    5:  "consumo de álcool e drogas; consumo de substâncias",
    6:  "violência; linguagem violento; linguagem vulgar e violento",
    7:  "relações familiares; relação mãe e filho; maternidade",
    9:  "mulheres; estereótipos e aparência das mulheres",
    11: "Vida em favela; bairros e favelas; cultura e favelas brasileiras",
    12: "arrependimento e pedidio de perdão; arrependimento religioso; arrpendimento e pedido de perdão aos pais",
}

SYSTEM_PROMPT = """Você é um analista de tópicos dentro de textos de música.

Considere a seguinte escala para realizar a comparação entre um tópico e um trecho:

1: Não há nenhuma relação semântica entre o trecho e o tópico.
2: A relação semântica é fraca entre o trecho e o tópico. Existe uma conexão mínima ou superficial entre o trecho e os temas discutidos no tópico, mas essa conexão não é particularmente evidente ou relevante.
3: A relação semântica é moderada entre o trecho e o tópico. O trecho compartilha algumas semelhanças ou temas gerais com o tópico, indicando uma conexão discernível, mas não muito substancial.
4: A relação semântica é forte entre o trecho e o tópico. O trecho está relacionado de forma significativa aos temas discutidos no tópico.
5: A relação semântica é muito forte entre o trecho e o tópico. O trecho está altamente alinhado com os temas e conceitos discutidos no tópico.

Responda exclusivamente um número entre 1 e 5"""


def build_prompt(trecho, num_topico):
    descricao = descricao_topicos[num_topico]
    current_few_shots = few_shot_prompt[str(num_topico)]
    positive_examples = current_few_shots['positivos']
    negative_examples = current_few_shots['negativos']

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT}
    ]

    # Add positive examples
    for example in positive_examples:
        messages.append({"role": "user", "content": f"Considere o tópico ({num_topico}): {descricao}\n\nTrecho: \"{example}\""})
        messages.append({"role": "assistant", "content": "5"}) # Assuming positive examples should have a score of 5

    # Add negative examples
    for example in negative_examples:
        messages.append({"role": "user", "content": f"Considere o tópico ({num_topico}): {descricao}\n\nTrecho: \"{example}\""})
        messages.append({"role": "assistant", "content": "1"}) # Assuming negative examples should have a score of 1

    # Add the actual user query
    messages.append({"role": "user", "content": f"Considere o tópico ({num_topico}): {descricao}\n\nTrecho: \"{trecho}\""})

    return messages


def score_trecho(trecho, num_topico):
    prompt = build_prompt(trecho, num_topico)
    response = get_response(prompt)
    return {
        "trecho": trecho,
        "num_topico": num_topico,
        "descricao": descricao_topicos[num_topico],
        "RESPONSE": response,
    }




# Teste
all_results = []
result = score_trecho(df["trecho"].iloc[0], num_topico=0)
all_results.append(result)
print(result)


Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


{'trecho': 'vai usar lily roupa da planet tommy joias louis vui vip e o paco rabanne', 'num_topico': 0, 'descricao': 'Consumo e ostentação de produto de luxos, marcas famosas; estil de vida luxoso', 'RESPONSE': '5'}


In [ ]:

with open('/content/drive/MyDrive/UFRGS/ESWA/few_shots_2026.pkl', 'rb') as file:
            all_results = pickle.load(file)
all_results = []

processed_pairs = set(
    (r['trecho'], r['num_topico']) for r in all_results
)

for trecho in df['trecho'].unique():
    for num_topico, descricao in descricao_topicos.items():
        # pula se já foi processado
        if (trecho, num_topico) in processed_pairs:
            print(f"Skipping (trecho, num_topico): ({trecho}, {num_topico})")
            continue

        response = score_trecho(trecho, num_topico=num_topico)
        print(response)

      #   result = {}
      #   result['trecho'] = trecho
      #   result['num_topico'] = num_topico
      #  # result['mean_score'] = aux_mean
      #   result['descricao'] = descricao
      #  # result['model'] = 'model'
      #   result['RESPONSE'] = response

        all_results.append(response)

        # Save the list to a file
        with open(base_path+'/few_shots_2026_v2.pkl', 'wb') as file:
            pickle.dump(all_results, file)

Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


{'trecho': 'vai usar lily roupa da planet tommy joias louis vui vip e o paco rabanne', 'num_topico': 0, 'descricao': 'Consumo e ostentação de produto de luxos, marcas famosas; estil de vida luxoso', 'RESPONSE': '5'}


Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


{'trecho': 'vai usar lily roupa da planet tommy joias louis vui vip e o paco rabanne', 'num_topico': 1, 'descricao': 'relacionamentos amorosos, intimidade, amor, paixão', 'RESPONSE': '1'}


Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


...

{'trecho': 'Um garoto doce e humilde que não desejava o mal à ninguém\nInfelizmente os sonhos foram pro alto, levou dois tiros e morreu num assalto\nSe não reagisse seria diferente, o sonho dele era estar aqui presente com a gente\nAo lado do trono de deus descansa uma pessoa rara\nJustiça seja feita, aqui se faz, aqui se paga', 'num_topico': 7, 'descricao': 'relações familiares; relação mãe e filho; maternidade', 'RESPONSE': '2'}


Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


{'trecho': 'Um garoto doce e humilde que não desejava o mal à ninguém\nInfelizmente os sonhos foram pro alto, levou dois tiros e morreu num assalto\nSe não reagisse seria diferente, o sonho dele era estar aqui presente com a gente\nAo lado do trono de deus descansa uma pessoa rara\nJustiça seja feita, aqui se faz, aqui se paga', 'num_topico': 9, 'descricao': 'mulheres; estereótipos e aparência das mulheres', 'RESPONSE': '1'}


Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


{'trecho': 'Um garoto doce e humilde que não desejava o mal à ninguém\nInfelizmente os sonhos foram pro alto, levou dois tiros e morreu num assalto\nSe não reagisse seria diferente, o sonho dele era estar aqui presente com a gente\nAo lado do trono de deus descansa uma pessoa rara\nJustiça seja feita, aqui se faz, aqui se paga', 'num_topico': 11, 'descricao': 'Vida em favela; bairros e favelas; cultura e favelas brasileiras', 'RESPONSE': '4'}
{'trecho': 'Um garoto doce e humilde que não desejava o mal à ninguém\nInfelizmente os sonhos foram pro alto, levou dois tiros e morreu num assalto\nSe não reagisse seria diferente, o sonho dele era estar aqui presente com a gente\nAo lado do trono de deus descansa uma pessoa rara\nJustiça seja feita, aqui se faz, aqui se paga', 'num_topico': 12, 'descricao': 'arrependimento e pedidio de perdão; arrependimento religioso; arrpendimento e pedido de perdão aos pais', 'RESPONSE': '4'}


'5'